
# Learning about the Gymnasium Library

In this notebook, that was created based on the [GitHub](https://github.com/PacktPublishing/Deep-Reinforcement-Learning-Hands-On-Third-Edition) of the Third Edition [Deep Reinforcement Learning Hands-On](https://www.oreilly.com/library/view/deep-reinforcement-learning/9781835882702/) book, you will learn the basics of [Gymnasium](https://gymnasium.farama.org/), a library used to provide a uniform API for RL agents and lots of RL environments.

You will try a first randomly behaving agent and become more familiar with the basic concepts of RL, in order to have an understanding of:
- The high-level requirements that need to be implemented to plug the agent into the RL framework
- A basic, pure-Python implementation of the random RL agent
- The Gymnasium library and its implementation

To render and show video with gym.wrappers.RecordVideo in Google Colab, it is required to install necessary libraries.

In [1]:
%%capture
#!pip install pyvirtualdisplay
#!apt-get install -y xvfb python-opengl ffmpeg

Let's first import a few dependencies we'll need.




In [2]:
import random
from typing import List

**Environment class**: Everything that is external to the agent and has the responsibility of providing observations and giving rewards. The environment changes its state based on the agent’s actions.

In [3]:
class Environment:
    def __init__(self):
        self.steps_left = 10

    def get_observation(self) -> List[float]:
        return [0.0, 0.0, 0.0]

    def get_actions(self) -> List[int]:
        return [0, 1]

    def is_done(self) -> bool:
        return self.steps_left == 0

    def action(self, action: int) -> float:
        if self.is_done():
            raise Exception("Game is over")
        self.steps_left -= 1
        return random.random()

**Agent class**: A thing, or person, that takes an active role. In practice, the agent is some piece of code that implements some policy. Basically, this policy decides what action is needed at every time step, given our observations.

In [4]:
class Agent:
    def __init__(self):
        self.total_reward = 0.0

    def step(self, env: Environment):
        current_observation = env.get_observation()
        actions = env.get_actions()
        reward = env.action(random.choice(actions))
        self.total_reward += reward

First execution with a random agent

In [5]:
# Create an environment
env = Environment()

# Create an agent
agent = Agent()

# Perform steps
while not env.is_done():
    agent.step(env)

print("Total reward got: %.4f" % agent.total_reward)

Total reward got: 5.8758


Creating the [Cart Pole](https://gymnasium.farama.org/environments/classic_control/cart_pole/) environment. A pole is attached by an un-actuated joint to a cart, which moves along a frictionless track. The pendulum is placed upright on the cart and the goal is to balance the pole by applying forces in the left and right direction on the cart.

In [6]:
def logStep(action, observation, reward, terminated, truncated, info):
    print(f"==== action: {action} ====")
    print(f"observation: {observation}\nreward: {reward}\nterminated: {terminated}\ntruncated: {truncated}\ninfo: {info}")

In [7]:
import gymnasium as gym

# Create a Gym environment
env = gym.make("CartPole-v1")
total_reward = 0.0
total_steps = 0

# Reset the environment
observation, info = env.reset()

# Perform steps and observe the reward
while True:
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)
    logStep(action, observation, reward, terminated, truncated, info)
    total_reward += reward
    total_steps += 1
    if terminated or truncated:
        break

print("Episode done in %d steps, total reward %.2f" % (total_steps, total_reward))

==== action: 1 ====
observation: [ 0.0427728   0.2144005  -0.02665139 -0.25649112]
reward: 1.0
terminated: False
truncated: False
info: {}
==== action: 1 ====
observation: [ 0.0470608   0.40989262 -0.03178121 -0.5574597 ]
reward: 1.0
terminated: False
truncated: False
info: {}
==== action: 0 ====
observation: [ 0.05525866  0.2152309  -0.04293041 -0.2749567 ]
reward: 1.0
terminated: False
truncated: False
info: {}
==== action: 0 ====
observation: [ 0.05956328  0.02074693 -0.04842954  0.00388281]
reward: 1.0
terminated: False
truncated: False
info: {}
==== action: 0 ====
observation: [ 0.05997821 -0.17364822 -0.04835188  0.28090093]
reward: 1.0
terminated: False
truncated: False
info: {}
==== action: 0 ====
observation: [ 0.05650525 -0.3680483  -0.04273387  0.5579503 ]
reward: 1.0
terminated: False
truncated: False
info: {}
==== action: 1 ====
observation: [ 0.04914428 -0.17235334 -0.03157486  0.25211596]
reward: 1.0
terminated: False
truncated: False
info: {}
==== action: 1 ====
observa

**RandomActionWrapper class**: Used to issue the random actions, making the agent to explore the environment and from time to time drift away from the beaten track of its policy.

In [8]:
class RandomActionWrapper(gym.ActionWrapper):
    def __init__(self, env: gym.Env, epsilon: float = 0.1):
        super(RandomActionWrapper, self).__init__(env)
        self.epsilon = epsilon

    def action(self, action: gym.core.WrapperActType) -> gym.core.WrapperActType:
        if random.random() < self.epsilon:
            action = self.env.action_space.sample()
            print(f"Random action {action}")
            return action
        return action

In [9]:
# Create a Gym environment with RandomActionWrapper
env = RandomActionWrapper(gym.make("CartPole-v1"))

# Reset the wrapped environment
observation, info = env.reset()
total_reward = 0.0

# Perform steps and observe the reward
while True:
    observation, reward, terminated, truncated, info = env.step(0)
    total_reward += reward
    logStep(0, observation, reward, terminated, truncated, info)
    if terminated or truncated:
        break

print(f"Reward got: {total_reward:.2f}")

==== action: 0 ====
observation: [ 0.03650163 -0.2079236   0.02509469  0.29480413]
reward: 1.0
terminated: False
truncated: False
info: {}
==== action: 0 ====
observation: [ 0.03234316 -0.40339416  0.03099077  0.5952947 ]
reward: 1.0
terminated: False
truncated: False
info: {}
Random action 0
==== action: 0 ====
observation: [ 0.02427527 -0.59893584  0.04289666  0.8975762 ]
reward: 1.0
terminated: False
truncated: False
info: {}
==== action: 0 ====
observation: [ 0.01229656 -0.79461217  0.06084819  1.2034285 ]
reward: 1.0
terminated: False
truncated: False
info: {}
==== action: 0 ====
observation: [-0.00359569 -0.9904657   0.08491676  1.5145434 ]
reward: 1.0
terminated: False
truncated: False
info: {}
==== action: 0 ====
observation: [-0.023405   -1.1865066   0.11520763  1.8324808 ]
reward: 1.0
terminated: False
truncated: False
info: {}
==== action: 0 ====
observation: [-0.04713514 -1.3827      0.15185724  2.158616  ]
reward: 1.0
terminated: False
truncated: False
info: {}
==== action

In [10]:
# Create a Gym environment that supports rgb_array rendering
env = gym.make("CartPole-v1", render_mode="rgb_array")

# Wrap the environment with RecordVideo to save videos
wrapped_env = gym.wrappers.RecordVideo(env, video_folder="./videos")

total_reward = 0.0
total_steps = 0

# Reset the wrapped environment, which will open the rendering window
observation, info = wrapped_env.reset()

# Perform steps and observe the rendering
while True:
    action = wrapped_env.action_space.sample()
    observation, reward, terminated, truncated, info = wrapped_env.step(action)
    logStep(action, observation, reward, terminated, truncated, info)
    total_reward += reward
    total_steps += 1
    if terminated or truncated:
        break

print(f"Episode done in {total_steps} steps, total reward {total_reward:.2f}")

# Close the environment to finalize video saving
wrapped_env.close()

/Users/gilcesarf/git/repositories/imd/imd3004-202602/.venv/lib/python3.14/site-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /Users/gilcesarf/git/repositories/imd/imd1103-202602/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


==== action: 0 ====
observation: [ 0.02796971 -0.14752692  0.02464226  0.29424396]
reward: 1.0
terminated: False
truncated: False
info: {}
==== action: 0 ====
observation: [ 0.02501918 -0.34299138  0.03052714  0.59459573]
reward: 1.0
terminated: False
truncated: False
info: {}
==== action: 0 ====
observation: [ 0.01815935 -0.538527    0.04241905  0.8967361 ]
reward: 1.0
terminated: False
truncated: False
info: {}
==== action: 1 ====
observation: [ 0.00738881 -0.344005    0.06035377  0.617683  ]
reward: 1.0
terminated: False
truncated: False
info: {}
==== action: 0 ====
observation: [ 5.087088e-04 -5.399158e-01  7.270744e-02  9.287474e-01]
reward: 1.0
terminated: False
truncated: False
info: {}
==== action: 1 ====
observation: [-0.01028961 -0.3458467   0.09128238  0.6597701 ]
reward: 1.0
terminated: False
truncated: False
info: {}
==== action: 0 ====
observation: [-0.01720654 -0.54211247  0.10447779  0.97974277]
reward: 1.0
terminated: False
truncated: False
info: {}
==== action: 1 ====

In [11]:
# from pyvirtualdisplay import Display

# # Virtual display
# display = Display(visible=0, size=(1400, 900))
# display.start()

In [12]:
from base64 import b64encode
from IPython.display import HTML

# Find the path to the recorded video (assuming one was recorded)
video_path = "./videos/rl-video-episode-0.mp4" # Adjust episode number if needed

# Gets a string containing a b4-encoded version of the MP4 video at the specified path
def show_video(video_path):
    mp4 = open(video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    return HTML(f'<video width="400" controls><source src="{data_url}" type="video/mp4"></video>')

In [13]:
show_video(video_path)

---

In [14]:
def getAction(observation):
    # observation = [cart_position, cart_velocity, pole_angle, pole_angular_velocity]
    cart_position = observation[0]
    cart_velocity = observation[1]
    pole_angle = observation[2]
    pole_angular_velocity = observation[3]

    # primeira aproximação olhando apenas velocidade do cart
    # if pole_angular_velocity > 0:
    #     action = 1  # pêndulo girando para a direita -> força para a direita
    # elif pole_angular_velocity < 0:
    #     action = 0  # pêndulo girando para a esquerda -> força para a esquerda
    # elif cart_velocity > 0:
    #     action = 0  # carrinho indo para a direita -> força para a esquerda
    # else:
    #     action = 1  # carrinho indo para a esquerda -> força para a direita
    # return action

    # segunda aproximação olhando apenas ângulo e velocidade angular do pêndulo
    score = pole_angle + 0.5 * pole_angular_velocity

    # terceira aproximação olhando posição e velocidade do cart e ângulo e velocidade angular do pêndulo
    # score = (
    #     -0.05 * cart_position
    #     -0.10 * cart_velocity
    #     +1.00 * pole_angle
    #     +0.50 * pole_angular_velocity
    # )

    if score > 0:
        action = 1  # direita
    else:
        action = 0  # esquerda
    
    return action
# Create a Gym environment that supports rgb_array rendering
env = gym.make("CartPole-v1", render_mode="rgb_array")

# Wrap the environment with RecordVideo to save videos
wrapped_env = gym.wrappers.RecordVideo(env, video_folder="./videos")

total_reward = 0.0
total_steps = 0

# Reset the wrapped environment, which will open the rendering window
observation, info = wrapped_env.reset()
action = getAction(observation)

# Perform steps and observe the rendering
while True:
    observation, reward, terminated, truncated, info = wrapped_env.step(action)

    action = getAction(observation)

    #logStep(action, observation, reward, terminated, truncated, info)
    total_reward += reward
    total_steps += 1
    if terminated or truncated:
        break

print(f"Episode done in {total_steps} steps, total reward {total_reward:.2f}")

# Close the environment to finalize video saving
wrapped_env.close()

Episode done in 500 steps, total reward 500.00


In [15]:
from base64 import b64encode
from IPython.display import HTML

# Find the path to the recorded video (assuming one was recorded)
video_path = "./videos/rl-video-episode-0.mp4" # Adjust episode number if needed

# Gets a string containing a b4-encoded version of the MP4 video at the specified path
def show_video(video_path):
    mp4 = open(video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    return HTML(f'<video width="400" controls><source src="{data_url}" type="video/mp4"></video>')

In [16]:
show_video(video_path)